In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
products_data = [
(101,"Rice Bag","Groceries","Hyderabad",1200,50),
(102,"Wheat Flour","Groceries","Bengaluru",900,80),
(103,"Sunflower Oil","Groceries","Mumbai",1800,40),
(104,"Milk Pack","Dairy","Chennai",60,200),
(105,"Cheese Block","Dairy","Delhi",450,70),
(106,"Soap","Personal Care","Kolkata",120,300),
(107,"Shampoo","Personal Care","Pune",320,150),
(108,"Toothpaste","Personal Care","Ahmedabad",90,250),
(109,"Notebook","Stationery","Hyderabad",75,500),
(110,"Pen Pack","Stationery","Mumbai",110,400),
(111,"LED TV","Electronics","Delhi",45000,15),
(112,"Refrigerator","Electronics","Chennai",38000,10),
(113,"Washing Machine","Electronics","Bengaluru",29000,12),
(114,"Mobile Phone","Electronics","Hyderabad",25000,35),
(115,"Laptop","Electronics","Pune",62000,18),
(116,"Air Conditioner","Electronics","Mumbai",42000,9),
(117,"Mixer Grinder","Home Appliances","Kolkata",3500,45),
(118,"Water Purifier","Home Appliances","Delhi",12000,20),
(119,"Ceiling Fan","Home Appliances","Ahmedabad",2800,60),
(120,"Gas Stove","Home Appliances","Chennai",5500,25)
]

products_columns = [
"product_id",
"product_name",
"category",
"warehouse_city",
"price",
"stock_quantity"
]

products_df = spark.createDataFrame(products_data, products_columns)

In [0]:
suppliers_data = [
(201,"Reddy Traders","Hyderabad","Groceries"),
(202,"Fresh Dairy Ltd","Chennai","Dairy"),
(203,"CarePlus Suppliers","Mumbai","Personal Care"),
(204,"Elite Electronics","Delhi","Electronics"),
(205,"OfficeKart","Bengaluru","Stationery"),
(206,"HomeNeeds Pvt Ltd","Pune","Home Appliances"),
(207,"National Grocers","Ahmedabad","Groceries"),
(208,"Smart Electronics","Kolkata","Electronics"),
(209,"Daily Essentials","Hyderabad","Personal Care"),
(210,"Kitchen World","Chennai","Home Appliances")
]

suppliers_columns = [
"supplier_id",
"supplier_name",
"supplier_city",
"specialization"
]

suppliers_df = spark.createDataFrame(suppliers_data, suppliers_columns)

In [0]:
orders_data = [
(301,101,201,"2024-04-01",20,"Delivered"),
(302,102,201,"2024-04-01",35,"Delivered"),
(303,111,204,"2024-04-02",2,"Delivered"),
(304,114,208,"2024-04-02",5,"Pending"),
(305,115,204,"2024-04-03",3,"Delivered"),
(306,104,202,"2024-04-03",50,"Delivered"),
(307,105,202,"2024-04-04",18,"Cancelled"),
(308,117,206,"2024-04-04",7,"Delivered"),
(309,118,210,"2024-04-05",4,"Pending"),
(310,119,206,"2024-04-05",12,"Delivered"),
(311,120,210,"2024-04-06",6,"Delivered"),
(312,113,204,"2024-04-06",4,"Delivered"),
(313,116,208,"2024-04-07",2,"Pending"),
(314,109,205,"2024-04-07",80,"Delivered"),
(315,110,205,"2024-04-08",120,"Delivered"),
(316,106,203,"2024-04-08",60,"Cancelled"),
(317,107,209,"2024-04-09",25,"Delivered"),
(318,108,203,"2024-04-09",40,"Delivered"),
(319,112,208,"2024-04-10",2,"Pending"),
(320,101,207,"2024-04-10",15,"Delivered")
]

orders_columns = [
"order_id",
"product_id",
"supplier_id",
"order_date",
"quantity",
"order_status"
]

orders_df = spark.createDataFrame(orders_data, orders_columns)

In [0]:
payments_data = [
(401,301,24000,"UPI","Paid"),
(402,302,31500,"Credit Card","Paid"),
(403,303,90000,"Bank Transfer","Paid"),
(404,304,125000,"UPI","Pending"),
(405,305,186000,"Bank Transfer","Paid"),
(406,306,3000,"Cash","Paid"),
(407,307,8100,"UPI","Cancelled"),
(408,308,24500,"Debit Card","Paid"),
(409,309,48000,"UPI","Pending"),
(410,310,33600,"Cash","Paid"),
(411,311,33000,"Credit Card","Paid"),
(412,312,116000,"Bank Transfer","Paid"),
(413,313,84000,"UPI","Pending"),
(414,314,6000,"Cash","Paid"),
(415,315,13200,"UPI","Paid"),
(416,316,7200,"Cash","Cancelled"),
(417,317,8000,"UPI","Paid"),
(418,318,3600,"Debit Card","Paid"),
(419,319,76000,"Bank Transfer","Pending"),
(420,320,18000,"UPI","Paid")
]

payments_columns = [
"payment_id",
"order_id",
"bill_amount",
"payment_mode",
"payment_status"
]

payments_df = spark.createDataFrame(payments_data, payments_columns)

In [0]:
orders_df = orders_df.withColumn(
    "order_date",
    to_date("order_date")
)

In [0]:
final_joined_df = products_df.join(
    orders_df,
    "product_id"
).join(
    suppliers_df,
    "supplier_id"
).join(
    payments_df,
    "order_id"
)

In [0]:
final_joined_df.createOrReplaceTempView(
    "final_joined_df"
)

In [0]:
display(final_joined_df)

61. Save final joined data as Delta table.

In [0]:
%sql
CREATE OR REPLACE TABLE final_orders
USING DELTA
AS
SELECT *
FROM final_joined_df

num_affected_rows,num_inserted_rows


62. Insert new order records.

In [0]:
%sql
INSERT INTO final_orders
VALUES
(
321,
114,
204,
'Mobile Phone',
'Electronics',
'Hyderabad',
25000,
35,
DATE('2024-04-11'),
3,
'Delivered',
'Elite Electronics',
'Delhi',
'Electronics',
421,
75000,
'UPI',
'Paid'
)

num_affected_rows,num_inserted_rows
1,1


63. Update one pending order.



In [0]:
%sql
UPDATE final_orders
SET order_status = 'Delivered'
WHERE order_id = 304

num_affected_rows
1


64. Update one pending payment.


In [0]:
%sql
UPDATE final_orders
SET payment_status = 'Paid'
WHERE payment_id = 404

num_affected_rows
1


65. Delete cancelled orders.


In [0]:
%sql
DELETE FROM final_orders
WHERE order_status = 'Cancelled'

num_affected_rows
2


66. Create clean_orders Delta table.


In [0]:
%sql
CREATE OR REPLACE TABLE clean_orders
USING DELTA
AS
SELECT *
FROM final_orders
WHERE order_status != 'Cancelled'

num_affected_rows,num_inserted_rows


67. Query Delta history.


In [0]:
%sql
DESCRIBE HISTORY final_orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-05-06T10:52:01.000Z,146342946703345,azuser5818_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3632705601078539),ad566dd9-bc72-41fb-b909-9ac526a4d34b,0506-104710-p8miongz-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 11073, p25FileSize -> 6336, numDeletionVectorsRemoved -> 1, minFileSize -> 6336, numAddedFiles -> 1, maxFileSize -> 6336, p75FileSize -> 6336, p50FileSize -> 6336, numAddedBytes -> 6336)",null,Databricks-Runtime/18.1.x-photon-scala2.13
5,2026-05-06T10:51:59.000Z,146342946703345,azuser5818_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(order_status#14056 = Cancelled)""])",null,List(3632705601078539),ad566dd9-bc72-41fb-b909-9ac526a4d34b,0506-104710-p8miongz-v2n,4,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1458, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 996, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 461)",null,Databricks-Runtime/18.1.x-photon-scala2.13
4,2026-05-06T10:51:43.000Z,146342946703345,azuser5818_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(payment_id#13314L = 404)""])",null,List(3632705601078539),56917b56-e8ac-49cd-823f-b5b77ff2c78a,0506-104710-p8miongz-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3029, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1095, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 4633, rewriteTimeMs -> 1934)",null,Databricks-Runtime/18.1.x-photon-scala2.13
3,2026-05-06T10:51:33.000Z,146342946703345,azuser5818_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3632705601078539),9f0748b3-5f16-4c6a-be0d-d007e0e11cec,0506-104710-p8miongz-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 10, numRemovedBytes -> 49484, p25FileSize -> 6440, numDeletionVectorsRemoved -> 1, minFileSize -> 6440, numAddedFiles -> 1, maxFileSize -> 6440, p75FileSize -> 6440, p50FileSize -> 6440, numAddedBytes -> 6440)",null,Databricks-Runtime/18.1.x-photon-scala2.13
2,2026-05-06T10:51:30.000Z,146342946703345,azuser5818_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(order_id#12263L = 304)""])",null,List(3632705601078539),9f0748b3-5f16-4c6a-be0d-d007e0e11cec,0506-104710-p8miongz-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 5268, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2641, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 4649, rewriteTimeMs -> 2586)",null,Databricks-Runtime/18.1.x-photon-scala2.13
1,2026-05-06T10:51:11.000Z,146342946703345,azuser5818_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3632705601078539),5f4ea63b-a23e-494b-8b8a-f3ef3fe8f271,0506-104710-p8miongz-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 4515)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-05-06T10:51:01.000Z,146342946703345,azuser5818_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVec

68. Query previous version using Time Travel.


In [0]:
%sql
SELECT *
FROM final_orders VERSION AS OF 0

order_id,supplier_id,product_id,product_name,category,warehouse_city,price,stock_quantity,order_date,quantity,order_status,supplier_name,supplier_city,specialization,payment_id,bill_amount,payment_mode,payment_status
313,208,116,Air Conditioner,Electronics,Mumbai,42000,9,2024-04-07,2,Pending,Smart Electronics,Kolkata,Electronics,413,84000,UPI,Pending
314,205,109,Notebook,Stationery,Hyderabad,75,500,2024-04-07,80,Delivered,OfficeKart,Bengaluru,Stationery,414,6000,Cash,Paid
315,205,110,Pen Pack,Stationery,Mumbai,110,400,2024-04-08,120,Delivered,OfficeKart,Bengaluru,Stationery,415,13200,UPI,Paid
308,206,117,Mixer Grinder,Home Appliances,Kolkata,3500,45,2024-04-04,7,Delivered,HomeNeeds Pvt Ltd,Pune,Home Appliances,408,24500,Debit Card,Paid
309,210,118,Water Purifier,Home Appliances,Delhi,12000,20,2024-04-05,4,Pending,Kitchen World,Chennai,Home Appliances,409,48000,UPI,Pending
310,206,119,Ceiling Fan,Home Appliances,Ahmedabad,2800,60,2024-04-05,12,Delivered,HomeNeeds Pvt Ltd,Pune,Home Appliances,410,33600,Cash,Paid
303,204,111,LED TV,Electronics,Delhi,45000,15,2024-04-02,2,Delivered,Elite Electronics,Delhi,Electronics,403,90000,Bank Transfer,Paid
304,208,114,Mobile Phone,Electronics,Hyderabad,25000,35,2024-04-02,5,Pending,Smart Electronics,Kolkata,Electronics,404,125000,UPI,Pending
305,204,115,Laptop,Electronics,Pune,62000,18,2024-04-03,3,Delivered,Elite Electronics,Delhi,Electronics,405,186000,Bank Transfer,Paid
318,203,108,Toothpaste,Personal Care,Ahmedabad,90,250,2024-04-09,40,Delivered,CarePlus Suppliers,Mumbai,Personal Care,418,3600,Debit Card,Paid


69. Compare old and latest versions.


In [0]:
%sql
SELECT *
FROM final_orders VERSION AS OF 0

In [0]:
%sql
SELECT *
FROM final_orders

70. Run VACUUM DRY RUN.

In [0]:
%sql
VACUUM final_orders
DRY RUN

In [0]:
daily_orders_data = [
(321,114,204,"2024-04-11",3,"Delivered"),
(322,118,210,"2024-04-11",2,"Delivered"),
(304,114,208,"2024-04-02",5,"Delivered"),
(319,112,208,"2024-04-10",2,"Delivered"),
(323,120,210,"2024-04-12",1,"Pending")
]
daily_orders_columns = [
"order_id",
"product_id",
"supplier_id",
"order_date",
"quantity",
"order_status"
]
daily_orders_df = spark.createDataFrame(
    daily_orders_data,
    daily_orders_columns
)

In [0]:
daily_orders_df = daily_orders_df.withColumn(
    "order_date",
    to_date("order_date")
)

71. Create Delta target table.


In [0]:
orders_df.createOrReplaceTempView(
    "orders_df"
)

In [0]:
%sql
CREATE OR REPLACE TABLE target_orders
USING DELTA
AS
SELECT *
FROM orders_df

num_affected_rows,num_inserted_rows


72. Load initial orders.


In [0]:
%sql
SELECT *
FROM target_orders

order_id,product_id,supplier_id,order_date,quantity,order_status
301,101,201,2024-04-01,20,Delivered
302,102,201,2024-04-01,35,Delivered
303,111,204,2024-04-02,2,Delivered
304,114,208,2024-04-02,5,Pending
305,115,204,2024-04-03,3,Delivered
306,104,202,2024-04-03,50,Delivered
307,105,202,2024-04-04,18,Cancelled
308,117,206,2024-04-04,7,Delivered
309,118,210,2024-04-05,4,Pending
310,119,206,2024-04-05,12,Delivered


73. Create temp view for updates.


In [0]:
daily_orders_df.createOrReplaceTempView(
    "daily_orders"
)

74. Merge updates into target table.

75. Update existing records.

76. Insert new records.


In [0]:
%sql
MERGE INTO target_orders AS t
USING daily_orders AS s
ON t.order_id = s.order_id

WHEN MATCHED THEN
UPDATE SET
t.product_id = s.product_id,
t.supplier_id = s.supplier_id,
t.order_date = s.order_date,
t.quantity = s.quantity,
t.order_status = s.order_status

WHEN NOT MATCHED THEN
INSERT
(
order_id,
product_id,
supplier_id,
order_date,
quantity,
order_status
)

VALUES
(
s.order_id,
s.product_id,
s.supplier_id,
s.order_date,
s.quantity,
s.order_status
)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,2,0,3


77. Verify updated order IDs.


In [0]:
%sql
SELECT *
FROM target_orders
WHERE order_id IN (304,319)

order_id,product_id,supplier_id,order_date,quantity,order_status
304,114,208,2024-04-02,5,Delivered
319,112,208,2024-04-10,2,Delivered


78. Verify inserted order IDs.


In [0]:
%sql
SELECT *
FROM target_orders
WHERE order_id IN (321,322,323)

order_id,product_id,supplier_id,order_date,quantity,order_status
321,114,204,2024-04-11,3,Delivered
322,118,210,2024-04-11,2,Delivered
323,120,210,2024-04-12,1,Pending


79. Check Delta history after merge.


In [0]:
%sql
DESCRIBE HISTORY target_orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-05-06T10:54:54.000Z,146342946703345,azuser5818_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(order_id#16168L = order_id#15274L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3632705601078539),61c346fc-d573-434a-a318-6269eb92f499,0506-104710-p8miongz-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 5, numTargetBytesAdded -> 9060, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3400, materializeSourceTimeMs -> 319, numTargetRowsInserted -> 3, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1207, numTargetRowsUpdated -> 2, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1813)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-05-06T10:54:35.000Z,146342946703345,azuser5818_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3632705601078539),2ac3e95a-3e31-44da-a2ae-77fa039b3551,0506-104710-p8miongz-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 20, numOutputBytes -> 2246)",null,Databricks-Runtime/18.1.x-photon-scala2.13


80. Explain why MERGE is important.

MERGE is important because it combines UPDATE and INSERT operations in a single statement.

Updates existing records automatically

Inserts new records automatically

Supports incremental data loading

Reduces duplicate records

Simplifies ETL and data pipeline processes

Improves data consistency in Delta tables

MERGE is widely used in:

Data Warehousing

Incremental Processing

Real-time data pipelines

Part 9 — Parquet to Delta

81. Save products data as Parquet.


In [0]:
products_df.write \
.mode("overwrite") \
.parquet("/tmp/products_parquet")

82. Read Parquet data.


In [0]:
parquet_df = spark.read.parquet(
    "/tmp/products_parquet"
)

In [0]:
display(parquet_df)

product_id,product_name,category,warehouse_city,price,stock_quantity
118,Water Purifier,Home Appliances,Delhi,12000,20
119,Ceiling Fan,Home Appliances,Ahmedabad,2800,60
120,Gas Stove,Home Appliances,Chennai,5500,25
108,Toothpaste,Personal Care,Ahmedabad,90,250
109,Notebook,Stationery,Hyderabad,75,500
110,Pen Pack,Stationery,Mumbai,110,400
113,Washing Machine,Electronics,Bengaluru,29000,12
114,Mobile Phone,Electronics,Hyderabad,25000,35
115,Laptop,Electronics,Pune,62000,18
103,Sunflower Oil,Groceries,Mumbai,1800,40


83. Convert Parquet into Delta.


In [0]:
%sql
CONVERT TO DELTA parquet.`/tmp/products_parquet`

84. Validate Delta table.


In [0]:
%sql
SELECT *
FROM delta.`/tmp/products_parquet`

85. Compare Parquet vs Delta behavior.

| Parquet              | Delta                      |
| -------------------- | -------------------------- |
| No UPDATE support    | Supports UPDATE            |
| No DELETE support    | Supports DELETE            |
| No Time Travel       | Supports Time Travel       |
| No ACID transactions | Supports ACID transactions |


86. Perform update on Delta table.


In [0]:
%sql
UPDATE delta.`/tmp/products_parquet`
SET price = 50000
WHERE product_id = 111

87. Explain why Delta supports updates better

Delta supports updates better because it provides ACID transactions, version history, and supports UPDATE, DELETE, and MERGE operations efficiently.

Part 11 — Unity Catalog and Governance

96. Create catalog.


In [0]:
%sql
CREATE CATALOG retail_catalog

97. Create schema.


In [0]:
%sql
CREATE SCHEMA retail_catalog.retail_schema

98. Register Delta table in Unity Catalog.


In [0]:
%sql
CREATE TABLE retail_catalog.retail_schema.final_orders
USING DELTA
AS
SELECT *
FROM final_orders

99. Create derived revenue table and inspect lineage.


In [0]:
%sql
CREATE OR REPLACE TABLE retail_catalog.retail_schema.category_revenue
AS
SELECT
category,
SUM(bill_amount) AS total_revenue
FROM retail_catalog.retail_schema.final_orders
GROUP BY category

In [0]:
%sql
DESCRIBE HISTORY retail_catalog.retail_schema.category_revenue

100. Apply SELECT permissions and explain governance behavior.

In [0]:
%sql
GRANT SELECT
ON TABLE retail_catalog.retail_schema.final_orders
TO `users`